In [2]:
import numpy as np
import cv2 
from math import sqrt
# create videocapture
cap = cv2.VideoCapture('output.mp4')

# create Background Subtractor objects
backSub = cv2.createBackgroundSubtractorKNN()

# Read Map Image
fmap = cv2.imread('2D_field.png')
fmap = cv2.resize(fmap,(1050,680))

# 9 correspondences for perspective transform
points1 = np.array([(144,166),
                    (1136,116),
                    (873,780),
                    (639,110),
                    (673,200),
                    (490,210),
                    (857,192),
                    (660,162),
                    (692,251)], dtype=np.int32)

points2 = np.array([(164,147),
                    (886,147),
                    (525,676),
                    (525,4),
                    (525,340),
                    (430,340),
                    (618,340),
                    (525,250),
                    (525,430)],   dtype=np.int32)   

# find perspective transform
H, _ = cv2.findHomography(points1, points2, cv2.RANSAC,5.0)

u=0 
first=0
centroids_p=[]
while True:
    
        
        # read video frame by frame
        ret, frame = cap.read()

        if ret == False:
            break

        if u%10==0:
            # apply GaussianBlur 
            m = 5
            frame2 = cv2.GaussianBlur(frame,(m,m),0)
            cv2.line(frame2,(0,120),(1280,57),color=(0,0,0),thickness=40)

            # update the background model (get foreground mask)
            fgMask = backSub.apply(frame2)

            # threshold for removing shadows
            ret, fgMask = cv2.threshold(fgMask,128,255,cv2.THRESH_BINARY)

            # opening
            # kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(7,15))

            kernel = np.array([ [0,0,0,1,1,0,0,0],
                                [0,0,0,1,1,0,0,0],
                                [0,1,1,1,1,1,1,0],
                                [0,1,1,1,1,1,1,0],
                                [0,1,1,1,1,1,1,0],
                                [0,1,1,1,1,1,1,0],
                                [0,0,1,1,1,1,0,0],
                                [0,0,1,1,1,1,0,0],
                                [0,0,1,1,1,1,0,0]], dtype='uint8')

            fgMask = cv2.morphologyEx(fgMask, cv2.MORPH_OPEN, kernel)


            #closing
            kernel = np.array([ [0,0,0,1,1,0,0,0],
                                [0,0,0,1,1,0,0,0],
                                [0,1,1,1,1,1,1,0],
                                [0,1,1,1,1,1,1,0],
                                [0,1,1,1,1,1,1,0],
                                [0,1,1,1,1,1,1,0],
                                [0,1,1,1,1,1,1,0],
                                [0,1,1,1,1,1,1,0],
                                [0,0,1,1,1,1,0,0],
                                [0,0,1,1,1,1,0,0],
                                [0,0,1,1,1,1,0,0]], dtype='uint8')

            fgMask = cv2.morphologyEx(fgMask, cv2.MORPH_CLOSE, kernel)

            n,C,stats, bad_centroids = cv2.connectedComponentsWithStats(fgMask);
            centroids=[]
            for i in range(1,n):
                point = bad_centroids[i].reshape(-1,1,2).astype(np.float32)
                dst = cv2.perspectiveTransform(point,H).reshape(1,2)
                dst = dst.astype('int32')
                centroids.append(dst)
                

            f = fmap.copy()

            
            for i in range(0,n-1):
                min=100000
                if stats[i,cv2.CC_STAT_TOP]<=240:
                    if stats[i,cv2.CC_STAT_AREA]>20:
                        dst2 = centroids[i]
                        for j in centroids_p:
                            if (dst2[0,0]-j[0,0])**2+(dst2[0,1]-j[0,1])**2<min:
                                min=(dst2[0,0]-j[0,0])**2+(dst2[0,1]-j[0,1])**2
                                tracked=(int(j[0,0]),int(j[0,1]))
                        if min<400 and first!=0:
                            cv2.line(f,tracked,(dst2[0,0], dst2[0,1]),color=(0,100,255),thickness=7)
                        cv2.circle(f,(dst2[0,0], dst2[0,1]), radius=7, color=[0,0,255], thickness=-1)
                        
                        
        #                 box = frame[stats[i,1]:stats[i,1]+stats[i,3],stats[i,0]:stats[i,0]+stats[i,2]]
        #                 box = cv2.resize(box,(45,45))
        #                 filename = str(u)+str(i)+'.jpg'
        #                 cv2.imwrite(filename,box)


                elif 240<stats[i,cv2.CC_STAT_TOP]<=480:
                    if stats[i,cv2.CC_STAT_AREA]>400:
                        dst2 = centroids[i]
                        for j in centroids_p:
                            if (dst2[0,0]-j[0,0])**2+(dst2[0,1]-j[0,1])**2<min:
                                min=(dst2[0,0]-j[0,0])**2+(dst2[0,1]-j[0,1])**2
                                tracked=(int(j[0,0]),int(j[0,1]))
                        if min<625 and first!=0:
                            cv2.line(f,tracked,(dst2[0,0], dst2[0,1]),color=(0,100,255),thickness=7)
                        cv2.circle(f,(dst2[0,0], dst2[0,1]), radius=7, color=[0,0,255], thickness=-1)
        #                 box = frame[stats[i,1]:stats[i,1]+stats[i,3],stats[i,0]:stats[i,0]+stats[i,2]]
        #                 box = cv2.resize(box,(45,45))
        #                 filename = str(u)+str(i)+'.jpg'
        #                 cv2.imwrite(filename,box)

                elif 480<stats[i,cv2.CC_STAT_TOP]:
                    if stats[i,cv2.CC_STAT_AREA]>1600:
                        dst2 = centroids[i]
                        for j in centroids_p:
                            if (dst2[0,0]-j[0,0])**2+(dst2[0,1]-j[0,1])**2<min :
                                min=(dst2[0,0]-j[0,0])**2+(dst2[0,1]-j[0,1])**2
                                tracked=(int(j[0,0]),int(j[0,1]))
                        if min<900 and first!=0:
                            cv2.line(f,tracked,(dst2[0,0], dst2[0,1]),color=(0,100,255),thickness=7)
                        cv2.circle(f,(dst2[0,0], dst2[0,1]), radius=7, color=[0,0,255], thickness=-1)
        #                 box = frame[stats[i,1]:stats[i,1]+stats[i,3],stats[i,0]:stats[i,0]+stats[i,2]]
        #                 box = cv2.resize(box,(45,45))
        #                 filename = str(u)+str(i)+'.jpg'
        #                 cv2.imwrite(filename,box)

            centroids_p=centroids
            first+=1
            # show the current frame ,the fg masks and Map
            cv2.imshow('Frame', frame)
            cv2.imshow('FG Mask', fgMask)
            cv2.imshow('F', f)
            

            if cv2.waitKey(100) == ord('q'):
                break
                
        u+=1
cap.release()
cv2.destroyAllWindows()    


C:\Users\AMIRHOSEIN\anaconda3\lib\site-packages\ipykernel_launcher.py:111: RuntimeWarning: overflow encountered in long_scalars
C:\Users\AMIRHOSEIN\anaconda3\lib\site-packages\ipykernel_launcher.py:112: RuntimeWarning: overflow encountered in long_scalars


dst2

In [218]:
for j in centroids_p:
                            if (dst2[0,0]-j[0,0])**2+(dst2[0,1]-j[0,1])**2<min:
                                min=(dst2[0,0]-j[0,0])**2+(dst2[0,1]-j[0,1])**2
                                tracked=(int(j[0,0]),int(j[0,1]))

In [219]:
tracked

(554, 470)

In [221]:
tracked

(554, 470)

In [164]:
first

890

In [165]:
u

170